In [1]:
! pip install confluent_kafka

# Pull 5 min data for sites PPA 27 and PPA 28

This notebook is used for Brendan to acquire 5 minute data for PPA 27 and 28. When production is not consistent, The notebook will smooth out the intervals (alpha version).

The output will be a csv file with two columns, datetime and metered production. The name of the file will be "{site name}\_{meter_id}\_{start_data}.csv"

Process
- connect to api and authenticate
- get sites and meters for CA sites
- change bin measurement to 5 min from 15
- call api
- save to csv file
- 

In [2]:
import logging

logger = logging.getLogger("site_fetcher")
logger.setLevel(logging.INFO)
if not logger.hasHandlers():
    handler = logging.StreamHandler()
    formatter = logging.Formatter('%(asctime)s %(levelname)s %(message)s')
    handler.setFormatter(formatter)
    logger.addHandler(handler)

In [3]:
import os
os.getcwd()

'/home/prod-user/solar_bi/site_fetcher'

In [4]:
# Example notebook cells

# Import necessary modules
import sys
sys.path.append('./producer')
import os
print(f'{os.getcwd()=}')

from producer.app import SiteFetcher
import requests
import pandas as pd

# # Initialize SiteFetcher with Kafka disabled
# fetcher = SiteFetcher(use_kafka=False)

# print(fetcher.password)
# # Test authentication
# cookie = fetcher.authenticate()
# print("Session cookie:", cookie)



os.getcwd()='/home/prod-user/solar_bi/site_fetcher'


In [5]:
ca_sites_dic = {
    '_28':{
        'name':'Pivot Energy PPA 28 LLC',
        'id':71309,
        'meter_id':474976,
    },
    '_27':{
    'name':'Pivot Energy PPA 27 LLC',
    'id':66982,
    'meter_id':475649,}
}
ca_sites_dic


{'_28': {'name': 'Pivot Energy PPA 28 LLC', 'id': 71309, 'meter_id': 474976},
 '_27': {'name': 'Pivot Energy PPA 27 LLC', 'id': 66982, 'meter_id': 475649}}

In [6]:


def get_5min_metered_production(site, hw_ids, start_date, end_date, metric='Production meter net energy'):
    
    metric_key = metric

    chart_id = fetcher.CUSTOM_CHART_IDS.get(metric_key)
    url = f"https://api.alsoenergy.com/Charts/Custom/{chart_id}/Data"
    
    headers = {
        'accept': 'application/json',
        'Cookie': fetcher.cookie
    }
    
    params = {
        "startTime": f"{start_date}T00:00:00",
        "endTime": f"{end_date}T00:00:00",
        "span": "Custom",
        "binSize": "Bin5Min",
        "aggregationMode": "BySite",
        "hardwareIds": str(hw_ids),
        "lineType": "Line",
        "selectedOptions": fetcher.SELECTED_OPTIONS_BY_METRIC.get(metric_key)
    }
    
    response = fetcher.session.request("GET", url, headers=headers, params=params)
    
    if response.status_code != requests.codes.ok:
        print(f"Error fetching custom metric data for {metric_key} from {site_id} {site_name} / {hw_ids} : {response.status_code}") 
        # return None
    
    payload = response.json()
    
    
    site_id = payload.get('site_id', -1)
    site_name = payload.get('site_name', ' ')
    raw_hw = payload.get("hardware_ids", "")
    hardware_ids = [int(x) for x in raw_hw.split(",") if x.strip()]
    
    series = next(
        (s for s in payload.get('data', [])
        if metric.lower() in s.get('title', '').lower()),
        None
    )
    
    df = pd.DataFrame(series['points'])
    units = series.get('yAxisKey', ' ')
    df.columns = ['time', units]
    return df



In [7]:
ca_site = '_28'

fetcher = SiteFetcher(use_kafka=False)
fetcher.authenticate()

start_date = '2025-08-07'
end_date = '2025-08-08'
metric = 'Production meter net energy'

for site in ca_sites_dic.keys():
    print(f"processing {site}...")
    site_id = ca_sites_dic[site]['id']
    hw_ids = ca_sites_dic[site]['meter_id']

    df = get_5min_metered_production(site_id, hw_ids, start_date, end_date, metric)

    name = ca_sites_dic[site]['name']

    df.to_csv(f"{name}_{hw_ids}_{start_date}.csv")

INFO:producer.app:Provided dates Start date: None, End date: None
INFO:producer.app:Initialized SiteFetcher with start date 2025-11-02 <class 'datetime.date'> and end date 2025-11-03 <class 'datetime.date'> with process type daily


Reading secrets from /app/secrets.json...
If this fails, ensure that the secrets file is mounted correctly in the container.
If running locally, ensure that the secrets file is in the correct location.
If running in production, ensure that the secrets file is mounted correctly in the container and that the path is correct.
Error reading secrets file:  [Errno 2] No such file or directory: '/app/secrets.json'
Attempting to read secrets file from ../secrets.json...
Kafka producer disabled, running without Kafka...
processing _28...
processing _27...


In [8]:
stop

NameError: name 'stop' is not defined